In [ ]:
!pip install -q delta-spark pyspark

In [2]:
import delta
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = SparkSession.builder \
    .appName("Part_8_merge") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()
print(delta.__version__)
print("Spark session ready")

4.2.0
Spark session ready


In [3]:
products_data = [
(101,"Rice Bag","Groceries","Hyderabad",1200,50),
(102,"Wheat Flour","Groceries","Bengaluru",900,80),
(103,"Sunflower Oil","Groceries","Mumbai",1800,40),
(104,"Milk Pack","Dairy","Chennai",60,200),
(105,"Cheese Block","Dairy","Delhi",450,70),
(106,"Soap","Personal Care","Kolkata",120,300),
(107,"Shampoo","Personal Care","Pune",320,150),
(108,"Toothpaste","Personal Care","Ahmedabad",90,250),
(109,"Notebook","Stationery","Hyderabad",75,500),
(110,"Pen Pack","Stationery","Mumbai",110,400),
(111,"LED TV","Electronics","Delhi",45000,15),
(112,"Refrigerator","Electronics","Chennai",38000,10),
(113,"Washing Machine","Electronics","Bengaluru",29000,12),
(114,"Mobile Phone","Electronics","Hyderabad",25000,35),
(115,"Laptop","Electronics","Pune",62000,18),
(116,"Air Conditioner","Electronics","Mumbai",42000,9),
(117,"Mixer Grinder","Home Appliances","Kolkata",3500,45),
(118,"Water Purifier","Home Appliances","Delhi",12000,20),
(119,"Ceiling Fan","Home Appliances","Ahmedabad",2800,60),
(120,"Gas Stove","Home Appliances","Chennai",5500,25)
]
products_columns = [
"product_id",
"product_name",
"category",
"warehouse_city",
"price",
"stock_quantity"
]
products_df = spark.createDataFrame(products_data, products_columns)

In [4]:
#Exercise 81
products_df.write.mode("overwrite").parquet("/tmp/products_parquet")

In [ ]:
#Exercise 82
products_parquet_df= spark.read.parquet("/tmp/products_parquet")
products_parquet_df.show()

In [6]:
#Exercise 83
products_parquet_df.write.format("delta").mode("overwrite").save("/tmp/products_delta")

In [ ]:
#Exercise 84
products_delta_df = spark.read.format("delta").load("/tmp/products_delta")
products_delta_df.show()

In [10]:
#Exercise 85
print(" Parquet count: " ,products_parquet_df.count())
print("Delta count: ", products_delta_df.count())

 Parquet count:  20
Delta count:  20


In [13]:
#Exercise 86
spark.sql(""" create table if not exists products_delta using delta
location '/tmp/products_delta' """)
spark.sql(""" update products_delta set price = price+1000
where category = "Electronics"  """)

DataFrame[num_affected_rows: bigint]

87. Delta supports updates better because it maintains a transaction log (_delta_log) that tracks all changes.
So updates/deletes are handled efficiently with ACID transactions, versioning, and time travel, unlike Parquet which is only file-based and does not track updates.